In [1]:
import os
import pandas as pd
from google.colab import userdata

# 1. Load Kaggle credentials from Colab Secrets
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# 2. Download the dataset directly from Kaggle
!kaggle datasets download -d suraj520/customer-support-ticket-dataset
!unzip -q customer-support-ticket-dataset.zip

# 3. Load the dataset and preview the relevant columns
df = pd.read_csv('customer_support_tickets.csv')

# We only need the text and the target tag for this project
df = df[['Ticket Description', 'Ticket Type']].dropna()

display(df.head())
print(f"Total tickets: {len(df)}")
print(f"Unique Categories: {df['Ticket Type'].unique()}")

Dataset URL: https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset
License(s): CC0-1.0
100% 828k/828k [00:01<00:00, 675kB/s]



,Ticket Description,Ticket Type
0,I'm having an issue with the {product_purchase...,Technical issue
1,I'm having an issue with the {product_purchase...,Technical issue
2,I'm facing a problem with my {product_purchase...,Technical issue
3,I'm having an issue with the {product_purchase...,Billing inquiry
4,I'm having an issue with the {product_purchase...,Billing inquiry


Total tickets: 8469
Unique Categories: ['Technical issue' 'Billing inquiry' 'Cancellation request'
 'Product inquiry' 'Refund request']


In [2]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.3 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Configure 4-bit memory-saving settings
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 2. Define the open-source model ID
model_id = "Qwen/Qwen2.5-3B-Instruct"

print("Downloading and loading the model... (This will take 1-2 minutes)")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("✨ Model loaded successfully on your Colab GPU!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✨ Model loaded successfully on your Colab GPU!


In [4]:
# Extract the unique tags from your loaded Kaggle dataset
AVAILABLE_TAGS = df['Ticket Type'].unique().tolist()

print(f"Your model will categorize tickets into these exact tags:\n{AVAILABLE_TAGS}")

Your model will categorize tickets into these exact tags:
['Technical issue', 'Billing inquiry', 'Cancellation request', 'Product inquiry', 'Refund request']


ZERO-SHOT-LEARNING

In [5]:
import json
import re

def local_zero_shot_classify(ticket_text, tags_list):
    # Construct an explicitly strict instruction set
    system_prompt = "You are an expert customer support routing assistant. You must respond ONLY with a valid JSON object."
    user_prompt = f"""Analyze the customer support ticket text below. Categorize it by choosing the top 3 most likely tags from this exact list: {tags_list}.

Ticket text: "{ticket_text}"

Your output must be structured exactly like this:
{{
  "top_3_tags": ["Tag1", "Tag2", "Tag3"]
}}
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Apply the model's specialized chat template framework
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate the token prediction
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=100,
            temperature=0.1,  # Kept very low to force strict structural adherence
            do_sample=False
        )

    # Clean and parse output text
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # Safely extract and load JSON from output string
    try:
        json_match = re.search(r"\{.*\}", response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))["top_3_tags"]
        return json.loads(response)["top_3_tags"]
    except Exception as e:
        return [f"Parsing Error. Raw text returned: {response}"]

ZERO-SHOT-LEARNING TESTING

In [6]:
# Sample 3 random rows from your real dataset
sample_df = df.sample(3, random_state=42)

for idx, row in sample_df.iterrows():
    ticket_content = row['Ticket Description']
    ground_truth = row['Ticket Type']

    # Get predictions
    predicted_top_3 = local_zero_shot_classify(ticket_content, AVAILABLE_TAGS)

    print("-" * 50)
    print(f"🎟️ Ticket Context: {ticket_content[:150]}...")
    print(f"✅ Ground Truth Label: {ground_truth}")
    print(f"🤖 LLM Top 3 Predictions: {predicted_top_3}")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--------------------------------------------------
🎟️ Ticket Context: I'm having an issue with the {product_purchased}. Please assist. I'm using xda-developer for something different. If there are issues with the {produc...
✅ Ground Truth Label: Refund request
🤖 LLM Top 3 Predictions: ['Technical issue', 'Product inquiry', 'Billing inquiry']
--------------------------------------------------
🎟️ Ticket Context: I'm having trouble connecting my {product_purchased} to my home Wi-Fi network. It doesn't detect any networks, although other devices are connecting f...
✅ Ground Truth Label: Product inquiry
🤖 LLM Top 3 Predictions: ['Technical issue', 'Product inquiry', 'Cancellation request']
--------------------------------------------------
🎟️ Ticket Context: I'm having an issue with the {product_purchased}. Please assist.

Please give credit to: @joeyclay I'm concerned about the security of my {product_pur...
✅ Ground Truth Label: Billing inquiry
🤖 LLM Top 3 Predictions: ['Technical issue',

In [4]:
import json
import torch
from tqdm import tqdm

top_1_zs = 0
top_3_zs = 0
total_samples = len(test_df)
failed_parses_zs = 0

print(f"🚀 Running Zero-Shot Baseline on {total_samples} test tickets...")

# The context manager dynamically turns off your fine-tuning for this block of code
with model.disable_adapter():
    for idx, row in tqdm(test_df.iterrows(), total=total_samples):
        ticket_text = row['Ticket Description']
        ground_truth = row['Ticket Type']

        messages = [
            {"role": "system", "content": "You are an expert customer support routing assistant. You must respond ONLY with a valid JSON object."},
            {"role": "user", "content": f'Analyze the customer support ticket text below. Categorize it by choosing the top 3 most likely tags from this exact list: {AVAILABLE_TAGS}.\n\nTicket text: "{ticket_text}"\n\nYour output must be structured exactly like this:\n{{\n  "top_3_tags": ["Tag1", "Tag2", "Tag3"]\n}}'}
        ]

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        try:
            parsed_json = json.loads(response)
            predicted_tags = parsed_json.get("top_3_tags", [])

            if len(predicted_tags) > 0 and predicted_tags[0] == ground_truth:
                top_1_zs += 1
            if ground_truth in predicted_tags:
                top_3_zs += 1
        except json.JSONDecodeError:
            failed_parses_zs += 1

# Calculate percentages
zs_top_1_acc = (top_1_zs / total_samples) * 100
zs_top_3_acc = (top_3_zs / total_samples) * 100

print("\n" + "="*40)
print("📊 ZERO-SHOT BASELINE RESULTS")
print("="*40)
print(f"🥇 Top-1 Accuracy: {zs_top_1_acc:.2f}%")
print(f"🥉 Top-3 Accuracy: {zs_top_3_acc:.2f}%")
print(f"⚠️ JSON Parse Failures: {failed_parses_zs}")
print("="*40)

🚀 Running Zero-Shot Baseline on 50 test tickets...


100%|██████████| 50/50 [01:18<00:00,  1.57s/it]


📊 ZERO-SHOT BASELINE RESULTS
🥇 Top-1 Accuracy: 22.00%
🥉 Top-3 Accuracy: 68.00%
⚠️ JSON Parse Failures: 0


FEW-SHOT-LEARNING


In [7]:
import random

# 1. Dynamically create our Few-Shot Examples from the dataframe
# We will grab 1 random example text for each unique category to teach the model.
few_shot_context = ""
for tag in AVAILABLE_TAGS:
    sample_match = df[df['Ticket Type'] == tag].sample(1, random_state=10).iloc[0]
    text_snippet = sample_match['Ticket Description'][:200] # keeps the prompt concise
    few_shot_context += f"Example Ticket: \"{text_snippet}...\"\nCorrect Category: \"{tag}\"\n\n"

# 2. Define the updated Few-Shot function
def local_few_shot_classify(ticket_text, tags_list, examples_string):
    system_prompt = "You are an expert customer support routing assistant. You must respond ONLY with a valid JSON object."

    user_prompt = f"""You are classifying customer support tickets into these categories: {tags_list}.
Study these reference examples carefully to understand how tickets are categorized:

{examples_string}

Now, classify the following new support ticket. Predict the top 3 most likely categories in order of probability.

New Ticket text: "{ticket_text}"

Your output must be structured exactly like this:
{{
  "top_3_tags": ["Tag1", "Tag2", "Tag3"]
}}
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=True # Fixed the warning by enabling sampling!
        )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    try:
        json_match = re.search(r"\{.*\}", response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))["top_3_tags"]
        return json.loads(response)["top_3_tags"]
    except Exception as e:
        return [f"Parsing Error: {response}"]

print("Few-shot text context successfully generated! Ready to test.")

Few-shot text context successfully generated! Ready to test.


FEW-SHOT-LEARNING TESTING


In [8]:
# Using the exact same seed to pull the same 3 sample rows as before
sample_df = df.sample(3, random_state=42)

print("--- RUNNING FEW-SHOT PREDICTIONS ---")
for idx, row in sample_df.iterrows():
    ticket_content = row['Ticket Description']
    ground_truth = row['Ticket Type']

    # Pass our newly minted context into the function
    predicted_top_3 = local_few_shot_classify(ticket_content, AVAILABLE_TAGS, few_shot_context)

    print("-" * 50)
    print(f"🎟️ Ticket Context: {ticket_content[:150]}...")
    print(f"✅ Ground Truth Label: {ground_truth}")
    print(f"🤖 LLM Few-Shot Predictions: {predicted_top_3}")

--- RUNNING FEW-SHOT PREDICTIONS ---
--------------------------------------------------
🎟️ Ticket Context: I'm having an issue with the {product_purchased}. Please assist. I'm using xda-developer for something different. If there are issues with the {produc...
✅ Ground Truth Label: Refund request
🤖 LLM Few-Shot Predictions: ['Technical issue', 'Product inquiry', 'Refund request']
--------------------------------------------------
🎟️ Ticket Context: I'm having trouble connecting my {product_purchased} to my home Wi-Fi network. It doesn't detect any networks, although other devices are connecting f...
✅ Ground Truth Label: Product inquiry
🤖 LLM Few-Shot Predictions: ['Technical issue', 'Product inquiry', 'Refund request']
--------------------------------------------------
🎟️ Ticket Context: I'm having an issue with the {product_purchased}. Please assist.

Please give credit to: @joeyclay I'm concerned about the security of my {product_pur...
✅ Ground Truth Label: Billing inquiry
🤖 LLM Fe

In [5]:
# 1. Dynamically build the few-shot example context from the training set
few_shot_context = ""
for tag in AVAILABLE_TAGS:
    sample_match = train_df[train_df['Ticket Type'] == tag].sample(1, random_state=42).iloc[0]
    text_snippet = sample_match['Ticket Description'][:200]
    few_shot_context += f"Example Ticket: \"{text_snippet}...\"\nCorrect Category: \"{tag}\"\n\n"

top_1_fs = 0
top_3_fs = 0
failed_parses_fs = 0

print(f"🚀 Running Few-Shot Baseline on {total_samples} test tickets...")

# Keep the fine-tuning adapter disabled for this baseline test
with model.disable_adapter():
    for idx, row in tqdm(test_df.iterrows(), total=total_samples):
        ticket_text = row['Ticket Description']
        ground_truth = row['Ticket Type']

        messages = [
            {"role": "system", "content": "You are an expert customer support routing assistant. You must respond ONLY with a valid JSON object."},
            {"role": "user", "content": f"""You are classifying customer support tickets into these categories: {AVAILABLE_TAGS}.
Study these reference examples carefully to understand how tickets are categorized:

{few_shot_context}

Now, classify the following new support ticket. Predict the top 3 most likely categories in order of probability.

New Ticket text: "{ticket_text}"

Your output must be structured exactly like this:
{{
  "top_3_tags": ["Tag1", "Tag2", "Tag3"]
}}"""}
        ]

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        try:
            parsed_json = json.loads(response)
            predicted_tags = parsed_json.get("top_3_tags", [])

            if len(predicted_tags) > 0 and predicted_tags[0] == ground_truth:
                top_1_fs += 1
            if ground_truth in predicted_tags:
                top_3_fs += 1
        except json.JSONDecodeError:
            failed_parses_fs += 1

# Calculate percentages
fs_top_1_acc = (top_1_fs / total_samples) * 100
fs_top_3_acc = (top_3_fs / total_samples) * 100

print("\n" + "="*40)
print("📊 FEW-SHOT BASELINE RESULTS")
print("="*40)
print(f"🥇 Top-1 Accuracy: {fs_top_1_acc:.2f}%")
print(f"🥉 Top-3 Accuracy: {fs_top_3_acc:.2f}%")
print(f"⚠️ JSON Parse Failures: {failed_parses_fs}")
print("="*40)

🚀 Running Few-Shot Baseline on 50 test tickets...


100%|██████████| 50/50 [01:21<00:00,  1.63s/it]


📊 FEW-SHOT BASELINE RESULTS
🥇 Top-1 Accuracy: 28.00%
🥉 Top-3 Accuracy: 62.00%
⚠️ JSON Parse Failures: 0


Fine-Tuning the LLM using QLoRA.

In [10]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [1]:
import os
import json
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

print("🔄 1. Loading and formatting dataset...")
df = pd.read_csv('customer_support_tickets.csv')[['Ticket Description', 'Ticket Type']].dropna()
AVAILABLE_TAGS = df['Ticket Type'].unique().tolist()

# Sample and split
mix_df = df.sample(450, random_state=42)
train_df = mix_df.head(400)

model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def format_row_to_chat(row):
    true_tag = row['Ticket Type']
    alt_tags = [t for t in AVAILABLE_TAGS if t != true_tag]
    simulated_top_3 = [true_tag, alt_tags[0], alt_tags[1]]

    messages = [
        {"role": "system", "content": "You are an expert customer support routing assistant. You must respond ONLY with a valid JSON object."},
        {"role": "user", "content": f'Analyze the customer support ticket text below. Categorize it by choosing the top 3 most likely tags from this exact list: {AVAILABLE_TAGS}.\n\nTicket text: "{row["Ticket Description"]}"\n\nYour output must be structured exactly like this:\n{{\n  "top_3_tags": ["Tag1", "Tag2", "Tag3"]\n}}'},
        {"role": "assistant", "content": f'{{"top_3_tags": {json.dumps(simulated_top_3)}}}'}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

hf_train_data = Dataset.from_pandas(train_df).map(format_row_to_chat)

print("\n🧠 2. Loading model in pure Float16 VRAM (Bypassing 4-bit limits)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("\n🔌 3. Attaching standard LoRA adapters...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)

print("\n🚀 4. Launching Fine-Tuning Loop...")
training_args = SFTConfig(
    output_dir="./qwen_support_model",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    max_steps=40,
    fp16=True,
    optim="adamw_torch",            # Native PyTorch optimizer
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_train_data,
    args=training_args,
)

trainer.train()
print("\n🎉 SUCCESS! Your fine-tuned model is fully trained.")

🔄 1. Loading and formatting dataset...


Map:   0%|          | 0/400 [00:00<?, ? examples/s]


🧠 2. Loading model in pure Float16 VRAM (Bypassing 4-bit limits)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

Adding EOS to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,2.042856
10,0.904993
15,0.691441
20,0.676964
25,0.635179
30,0.505715
35,0.533049
40,0.516280



🎉 SUCCESS! Your fine-tuned model is fully trained.


FINE-TUNNING-TESTING

In [2]:
import torch
import json

# 1. Recreate the exact 50 test rows from our evaluation split
mix_df = df.sample(450, random_state=42)
test_df = mix_df.tail(50)

# 2. Put model in evaluation mode for stable inference
model.eval()

print("🚀 Running predictions on holdout test data...\n")

# Evaluate the first 3 test items to visually inspect the JSON structure
for idx, row in test_df.head(3).iterrows():
    ticket_text = row['Ticket Description']
    ground_truth = row['Ticket Type']

    # Construct the exact same prompt structure used during training
    messages = [
        {"role": "system", "content": "You are an expert customer support routing assistant. You must respond ONLY with a valid JSON object."},
        {"role": "user", "content": f'Analyze the customer support ticket text below. Categorize it by choosing the top 3 most likely tags from this exact list: {AVAILABLE_TAGS}.\n\nTicket text: "{ticket_text}"\n\nYour output must be structured exactly like this:\n{{\n  "top_3_tags": ["Tag1", "Tag2", "Tag3"]\n}}'}
    ]

    # Render the text, adding the generation prompt so the model knows it's its turn to speak
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,    # Low temperature keeps outputs sharp and consistent
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # Strip the prompt text and decode only the fresh response tokens
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    print(f"🎟️ Ticket Context: {ticket_text[:140]}...")
    print(f"✅ Ground Truth Label: {ground_truth}")
    print(f"🤖 Fine-Tuned Model Generation:\n{response}")
    print("-" * 60)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🚀 Running predictions on holdout test data...

🎟️ Ticket Context: I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it...
✅ Ground Truth Label: Refund request
🤖 Fine-Tuned Model Generation:
{"top_3_tags": ["Product inquiry", "Technical issue", "Billing inquiry"]}
------------------------------------------------------------
🎟️ Ticket Context: I've recently set up my {product_purchased}, but it fails to connect to any available networks. What steps should I take to troubleshoot thi...
✅ Ground Truth Label: Technical issue
🤖 Fine-Tuned Model Generation:
{"top_3_tags": ["Technical issue", "Billing inquiry", "Cancellation request"]}
------------------------------------------------------------
🎟️ Ticket Context: I'm having an issue with the {product_purchased}. Please assist.

Click to expand... This problem started occurring after the recent softwar...
✅ Ground Truth Label: Refund request
🤖 Fine-Tuned Mo

In [3]:
import json
import torch
from tqdm import tqdm

model.eval()

top_1_correct = 0
top_3_correct = 0
total_samples = len(test_df)
failed_parses = 0

print(f"🚀 Evaluating {total_samples} test tickets. This will take a minute...\n")

for idx, row in tqdm(test_df.iterrows(), total=total_samples):
    ticket_text = row['Ticket Description']
    ground_truth = row['Ticket Type']

    # Rebuild the prompt
    messages = [
        {"role": "system", "content": "You are an expert customer support routing assistant. You must respond ONLY with a valid JSON object."},
        {"role": "user", "content": f'Analyze the customer support ticket text below. Categorize it by choosing the top 3 most likely tags from this exact list: {AVAILABLE_TAGS}.\n\nTicket text: "{ticket_text}"\n\nYour output must be structured exactly like this:\n{{\n  "top_3_tags": ["Tag1", "Tag2", "Tag3"]\n}}'}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate the prediction
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Extract and score the JSON
    try:
        parsed_json = json.loads(response)
        predicted_tags = parsed_json.get("top_3_tags", [])

        # Check Top-1
        if len(predicted_tags) > 0 and predicted_tags[0] == ground_truth:
            top_1_correct += 1

        # Check Top-3
        if ground_truth in predicted_tags:
            top_3_correct += 1

    except json.JSONDecodeError:
        failed_parses += 1

# Calculate final percentages
top_1_acc = (top_1_correct / total_samples) * 100
top_3_acc = (top_3_correct / total_samples) * 100

print("\n\n" + "="*40)
print("🎯 FINAL EVALUATION RESULTS")
print("="*40)
print(f"🥇 Top-1 Accuracy: {top_1_acc:.2f}% ({top_1_correct}/{total_samples})")
print(f"🥉 Top-3 Accuracy: {top_3_acc:.2f}% ({top_3_correct}/{total_samples})")
print(f"⚠️ JSON Parse Failures: {failed_parses}")
print("="*40)

🚀 Evaluating 50 test tickets. This will take a minute...



100%|██████████| 50/50 [01:55<00:00,  2.31s/it]



🎯 FINAL EVALUATION RESULTS
🥇 Top-1 Accuracy: 24.00% (12/50)
🥉 Top-3 Accuracy: 58.00% (29/50)
⚠️ JSON Parse Failures: 0


Save Weights

In [6]:
import shutil
from google.colab import drive

# 1. Explicitly save the final trained adapter and tokenizer files to the local folder
print("Saving model files locally...")
trainer.save_model("./qwen_support_model")
tokenizer.save_pretrained("./qwen_support_model")

# 2. Mount your Google Drive into this Colab session
print("\nMounting Google Drive... (Please approve the pop-up permission window)")
drive.mount('/content/drive')

# 3. Copy the weights folder directly into your main Google Drive directory
source_folder = "./qwen_support_model"
destination_folder = "/content/drive/MyDrive/qwen_support_model"

print("\nCopying files to your Google Drive... This will take just a few seconds.")
shutil.copytree(source_folder, destination_folder, dirs_exist_ok=True)

print(f"\n🎉 SUCCESS! Your custom fine-tuned weights are safely backed up to your Google Drive at: {destination_folder}")

Saving model files locally...

Mounting Google Drive... (Please approve the pop-up permission window)
Mounted at /content/drive

Copying files to your Google Drive... This will take just a few seconds.

🎉 SUCCESS! Your custom fine-tuned weights are safely backed up to your Google Drive at: /content/drive/MyDrive/qwen_support_model
